# 3. Five strategies, one test

The golden cross did not beat holding, and a model could not rescue it. So this notebook puts four other well known ideas through the same test.

The list, the settings and the scoring were fixed in [`PLAN.md`](../PLAN.md) before anything was run. The years from 2022 on were cut off from every strategy until one final run.

In [ ]:
import os, sys, json
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams.update({"figure.facecolor": "black", "axes.facecolor": "black", "axes.edgecolor": "#3a3b3e", "axes.grid": True,
                     "grid.color": "#1c1c1e", "figure.figsize": (11, 4.5), "axes.spines.top": False, "axes.spines.right": False})
WHITE, SILVER, GREY, JADE = "#f2f2f0", "#d4d6da", "#8d8d90", "#5fd0a0"
pd.options.display.float_format = "{:.3f}".format

In [ ]:
report = json.load(open("results/strategies_final.json"))
daily = pd.read_csv("results/strategies_final_daily.csv", index_col=0, parse_dates=True)
dev, final = daily.loc[:"2021"], daily.loc["2022":]
COLORS = {"Hold everything": SILVER, "Golden cross": "#6f7074", "Momentum": JADE, "Market trend filter": WHITE, "Dip buying": "#4a4b4f", "SPY": GREY}

def show(period):
    table = pd.DataFrame(report[period]).T
    table.columns = ["Yearly return", "Worst drop", "Yearly swing", "Return per unit of swing"]
    return table.style.format({"Yearly return": "{:.1%}", "Worst drop": "{:.0%}", "Yearly swing": "{:.0%}", "Return per unit of swing": "{:.2f}"})

def growth_chart(part, title):
    fig, ax = plt.subplots()
    for name in part:
        ax.plot((1 + part[name]).cumprod(), color=COLORS[name], lw=1.8 if name in ("Momentum", "Market trend filter") else 1.2, ls=":" if name == "SPY" else "-", label=name)
    ax.set_yscale("log"); ax.set_ylabel("Value of 1 dollar"); ax.set_title(title, loc="left"); ax.legend(frameon=False, ncol=2)
    plt.show()

## Development years: 2005 to 2021

These were looked at first. **Return per unit of swing** is the yearly return divided by how much the value bounces around. Higher means more reward for the same amount of risk.

In [ ]:
show("development 2005 to 2021")

In [ ]:
growth_chart(dev, "Development years")

On these years the market trend filter looks like the winner. It earned nearly as much as holding with less than half the worst drop. Momentum trailed plain holding. If the story ended here, the trend filter would be the strategy to build on.

## The final test: 2022 to today

Run once, with nothing changed.

In [ ]:
show("final test 2022 to today")

In [ ]:
growth_chart(final, "Final test years")

## Side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, column, label in [(axes[0], "yearly_return", "Yearly return"), (axes[1], "worst_drop", "Worst drop")]:
    table = pd.DataFrame({"Development": pd.DataFrame(report["development 2005 to 2021"]).T[column], "Final test": pd.DataFrame(report["final test 2022 to today"]).T[column]})
    x = np.arange(len(table))
    ax.bar(x - 0.2, table["Development"], 0.4, color=SILVER, label="Development")
    ax.bar(x + 0.2, table["Final test"], 0.4, color=JADE, label="Final test")
    ax.set_xticks(x, table.index, rotation=30, ha="right"); ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.set_title(label, loc="left")
axes[0].legend(frameon=False)
plt.tight_layout(); plt.show()

## Falls from the peak, across the whole period

In [ ]:
fig, ax = plt.subplots()
for name in ["Hold everything", "Momentum", "Market trend filter", "Golden cross"]:
    value = (1 + daily[name]).cumprod()
    ax.plot(value / value.cummax() - 1, color=COLORS[name], lw=1.2, label=name)
ax.axvline(pd.Timestamp("2022-01-01"), color=GREY, lw=0.8, ls=":")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.legend(frameon=False, loc="lower right")
plt.show()

## What this says

* **The development winner lost the final test.** The market trend filter looked best through 2021, then earned 6 percent a year against 14 for holding. It stepped out during the 2022 decline and came back late, and its worst drop was no better than holding.
* **The development laggard won it.** Momentum trailed holding through 2021, then earned 20 percent a year in the final test.
* **Nothing beat holding in both periods.** Every strategy that looked good in one period looked ordinary or poor in the other.
* **The golden cross did what it always does.** Lower return, smaller drops, in both periods.
* **Dip buying was the weakest throughout.**

This is the most useful result in the project. Had the choice been made on the development years alone, the pick would have been the trend filter, and it would have disappointed. Holding back the final years is what exposed that.

The same limits apply as before: price history is missing for about a third of past index members, mostly ones that failed, and cash earns nothing here.